# Importación de librerías

In [1]:
from pathlib import Path

from langchain_core.documents.base import Document
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

e:\proyecto\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Importación de datos

Recogemos todo el corpus de la carpeta `processed/corpus`.

In [2]:
ROOT_DIR = Path("E:/proyecto")
CORPUS_DIR = ROOT_DIR / "data" / "processed" / "corpus"

docs = []
for doc_path in sorted(CORPUS_DIR.rglob("*.md")):
    species = doc_path.relative_to(CORPUS_DIR).parts[0]

    with open(doc_path, 'r', encoding='utf-8') as f:
        text = f.read()

    doc = Document(
        page_content=text,
        metadata = {
            "species_name": species.replace("_", " ").capitalize() # Lo dejamos con el nombre "correcto" de la planta para facilitar la búsqueda de datos.
        }
    )

    docs.append(doc)

print(f"Importados {len(docs)} documentos.")

Importados 91 documentos.


Realizamos la división en chunks de los datos utilizando `RecursiveCharacterTextSplitter`. En este caso, creamos divisiones de 500 caracteres con 100 caracteres de overlap.

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100
)

chunked_docs = splitter.split_documents(docs)
print(f"Dividido en {len(chunked_docs)} chunks.")

Dividido en 527 chunks.


# Creación de base de datos vectorial

Cargamos nuestro modelo para realizar los embeddings.

In [4]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-qwen3-embedding-0.6b",
    base_url="http://localhost:1234/v1",
    api_key="lm-studio-dummy",
    check_embedding_ctx_length=False
)

print("Modelo de embeddings cargado.")

Modelo de embeddings cargado.


Y creamos nuestra base de datos vectorial utilizando Chroma.

In [5]:
vector_store = Chroma.from_documents(
    documents = chunked_docs,
    embedding = embeddings,
    persist_directory = str(ROOT_DIR / "chroma_db")
)

print("Base de datos de vectores creada correctamente.")

Base de datos de vectores creada correctamente.


# Probando el RAG

In [ ]:
vector_store.similarity_search_with_score("Háblame del Dracaena draco")

[(Document(id='aa893582-8740-49d8-8ece-6373a4128067', metadata={'species_name': 'Euphorbia lamarckii'}, page_content='## Flores'),
  0.5810930728912354),
 (Document(id='039d9732-3044-4272-97a3-a9cbe3f9b4b7', metadata={'species_name': 'Rumex lunaria'}, page_content='## Flores'),
  0.5810930728912354),
 (Document(id='aeda1a1e-e195-42f6-83ee-edd3a44c4ca4', metadata={'species_name': 'Rumex lunaria'}, page_content='## Flores'),
  0.5810930728912354),
 (Document(id='906baa77-fc97-4537-a782-fa220c84791a', metadata={'species_name': 'Sonchus canariensis'}, page_content='## Flores'),
  0.5810930728912354)]